In [1]:
#dependencias
import numpy as np
import pandas as pd
import csv
from sklearn.preprocessing import StandardScaler

### Datos Originales de Tráfico de Red
El desarrollo comienza a partir del conjunto de datos MedSec-25 de Kaggle, el cual registra eventos de tráfico e red en dispositivos de IoT Médico (IoMT). Cada registro representa una captura de tráfico individual con sus características técnicas y de red.

In [2]:
#Lectura de datos desde Kaggle
import kagglehub
from kagglehub import KaggleDatasetAdapter

#nombre del archivo original
file_path = "MedSec-25.csv"

#cagar la última version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "abdullah001234/medsec-25-iomt-cybersecurity-dataset",
  file_path,
)

print("Lectura exitosa")

c:\Users\usr\Documents\LCD\Investigación operativa 2\HMM-Ciberseguridad\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\usr\AppData\Local\Temp\ipykernel_6536\1337715427.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Lectura exitosa


### Visualización general de los datos

In [3]:
#variable objetivo: Label
df['Label'].unique()

<ArrowStringArray>
[          'Benign',     'Exfiltration',   'Initial access',
 'Lateral movement',   'Reconnaissance']
Length: 5, dtype: str

In [4]:
df.columns

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts',
       'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
       'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std',
       'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean',
       'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean',
       'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot',
       'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
       'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',
       'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags',
       'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s',
       'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean',
       'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt',
       'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt',
       'CWE Flag Count', 'ECE 

### Definir las variables observables
Son las tres métricas cuantitativas continuas del flujo que caracterizan el comportamiento de la red
- ``Flow Duration``: Duración total del flujo de datos en microsegundos.
- ``Tot Fwd Pkts``: Cantidad total de paquetes enviados hacia adelante (Forward).
- ``Tot Bwd Pkts``: Cantidad total de paquetes enviados hacia atrás (Backward).

Variables contextuales y temporales:

- ``Src IP`` y ``Dst IP``: Direcciones IP de origen y destino del tráfico.

- ``Timestamp``: Marca de tiempo exacta (fecha y hora en que ocurrió el evento).

- ``Label``: Etiqueta del estado real de la red (Benign, Initial access, Reconnaissance, Lateral movement, Exfiltration) utilizada para validación posterior del modelo.

In [5]:
var_observables = [
    "Flow Duration",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
    "Src IP",
    "Dst IP",
    "Timestamp",
    "Label"
]

df_filtrado = df[var_observables].copy()

In [6]:
df_filtrado.head()

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,Src IP,Dst IP,Timestamp,Label
0,119182335,119,1,127.0.0.1,127.0.0.1,31/05/2022 12:36:00 AM,Benign
1,119182336,119,1,127.0.0.1,127.0.0.1,31/05/2022 12:36:00 AM,Benign
2,119952116,473,475,192.168.1.102,192.168.1.101,31/05/2022 12:36:00 AM,Benign
3,119769560,236,1,127.0.0.1,127.0.0.1,31/05/2022 12:36:01 AM,Benign
4,119769549,236,1,127.0.0.1,127.0.0.1,31/05/2022 12:36:01 AM,Benign


In [7]:
df_filtrado.info()

<class 'pandas.DataFrame'>
RangeIndex: 554534 entries, 0 to 554533
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Flow Duration  554534 non-null  int64
 1   Tot Fwd Pkts   554534 non-null  int64
 2   Tot Bwd Pkts   554534 non-null  int64
 3   Src IP         554534 non-null  str  
 4   Dst IP         554534 non-null  str  
 5   Timestamp      554534 non-null  str  
 6   Label          554534 non-null  str  
dtypes: int64(3), str(4)
memory usage: 61.9 MB


### Limpieza de datos
> ##### Descarto valores negativos, presentes en `Flow Duration` 
> ##### Transformo `Timestamp` a tipo `Datetime`

In [8]:
df_limpio = df_filtrado[df_filtrado['Flow Duration'] >= 0].copy()

#transformación de timestamp a datetime
df_limpio['Timestamp'] = pd.to_datetime(df_limpio['Timestamp'], format='%d/%m/%Y %I:%M:%S %p')
# df_limpio['Timestamp'] = df_limpio['Timestamp'].dt.strftime('%d/%m/%Y')

In [9]:
#chequeo el correcto procesamiento
df_limpio.head()

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,Src IP,Dst IP,Timestamp,Label
0,119182335,119,1,127.0.0.1,127.0.0.1,2022-05-31 00:36:00,Benign
1,119182336,119,1,127.0.0.1,127.0.0.1,2022-05-31 00:36:00,Benign
2,119952116,473,475,192.168.1.102,192.168.1.101,2022-05-31 00:36:00,Benign
3,119769560,236,1,127.0.0.1,127.0.0.1,2022-05-31 00:36:01,Benign
4,119769549,236,1,127.0.0.1,127.0.0.1,2022-05-31 00:36:01,Benign


### Reconstrucción de la Secuencia Temporal

Un Modelo Oculto de Markov (HMM) requiere analizar cómo evoluciona un proceso en el tiempo de manera secuencial. Para lograr esto sin romper la lógica del tráfico de red:

- **Agrupación por Canal**: Se agrupó el dataset según la combinación de dirección IP de origen (Src IP) y de destino (Dst IP). Cada par representa una sesión o canal de comunicación real entre dos dispositivos.

- **Orden Cronológico**: Dentro de cada sesión, se ordenaron todos los eventos en orden temporal estricto de menor a mayor mediante la fecha y hora exacta de Timestamp.

- **Cálculo de Longitudes de Secuencia**: Se contó cuántos eventos consecutivos ocurrieron en cada canal de comunicación. Esto genera un vector de longitudes (longitud), donde cada elemento indica exactamente cuántas observaciones continuas forman una conversación en la red.

In [10]:
df_ordenado = df_limpio.sort_values(by=['Src IP', 'Dst IP', 'Timestamp']).reset_index(drop=True)

In [11]:
df_ordenado.head()

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,Src IP,Dst IP,Timestamp,Label
0,2867254,1,1,0.0.0.0,255.255.255.255,2025-04-23 12:35:35,Exfiltration
1,438769,1,1,0.0.0.0,255.255.255.255,2025-04-24 16:01:09,Lateral movement
2,18,3,1,10.0.2.15,10.0.2.3,2025-04-17 15:42:26,Initial access
3,11,3,1,10.0.2.15,10.0.2.3,2025-04-17 15:42:30,Initial access
4,5006219,7,1,10.0.2.15,10.0.2.3,2025-04-17 15:42:32,Initial access


In [12]:
#guardo en un .csv
df_ordenado.to_csv('../datos_procesados/original_ordenado.csv', index=False)

In [13]:
tam_grupos = df_ordenado.groupby(['Src IP', 'Dst IP'], sort=False).size()
longitud = tam_grupos.tolist()

In [14]:
len(longitud)

268

### Acondicionamiento Estadístico

Las variables continuas originales presentan distribuciones fuertemente asimétricas y rangos de magnitud muy dispersos (duraciones de microsegundos frente a conteos de paquetes pequeños). Para evitar inestabilidades numéricas al entrenar el HMM, se aplicó un doble acondicionamiento:

- **Transformación Logarítmica** ($\ln(x + 1)$): Suaviza la asimetría de los datos y comprime las amplitudes extremas sin alterar la relación de orden entre las mediciones.
- **Estandarización** ($StandardScaler$): Escala los valores logarítmicos para que cada variable tenga media $0$ y desviación estándar $1$. Esto equilibra el peso de las tres variables al calcular las distribuciones de emisión del HMM.

In [15]:
obs_cols = ['Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts']
X_log = np.log1p(df_ordenado[obs_cols])

scaler = StandardScaler()
sec_escaladas = scaler.fit_transform(X_log)

In [16]:
# Guardar la matriz de datos sec_escaladas
df_escalado = pd.DataFrame(sec_escaladas, columns=['Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts'])
df_escalado.to_csv('../datos_procesados/secuencias.csv', index=False)

#guardo las longitudes 
df_longitud = pd.DataFrame({'longitud': longitud})
df_longitud.to_csv('../datos_procesados/longitud_secuencias.csv', index=False)

#### Al finalizar este proceso, se obtuvieron los tres insumos estructurados y alineados necesarios para ajustar el modelo de forma no supervisada:

1. Tabla de secuencias ordenadas: `original_ordenado.csv`

2. Matriz de observaciones: `secuencias.csv`

3. Vector de longitudes: `longitudes_secuencias.csv`